YOLOv5 Git Clone

In [1]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -qr requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17851, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 17851 (delta 33), reused 16 (delta 9), pack-reused 17798 (from 2)
Receiving objects: 100% (17851/17851), 17.00 MiB | 19.05 MiB/s, done.
Resolving deltas: 100% (12169/12169), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 212.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 468.7 kB/s eta 0:00:00


Import Libraries

In [2]:
import torch
import os
import cv2
import numpy as np
import random
import shutil
from pathlib import Path
from google.colab import files
import matplotlib.pyplot as plt

Dataset Upload

In [3]:
import zipfile

print("Upload YOLOv5 dataset")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

raw_dir = "/content/dataset_raw"
os.makedirs(raw_dir, exist_ok=True)
with zipfile.ZipFile(zip_filename, 'r') as z:
    z.extractall(raw_dir)

all_images = sorted([f for f in Path(raw_dir).rglob('*.jpg')])
print(f"{len(all_images)} images found")

random.seed(42)
random.shuffle(all_images)
split = int(len(all_images) * 0.8)
train_imgs = all_images[:split]
val_imgs   = all_images[split:]
print(f"{len(train_imgs)} images for training and {len(val_imgs)} images for validation")

for split_name in ['train', 'val']:
    os.makedirs(f"/content/dataset/{split_name}/images", exist_ok=True)
    os.makedirs(f"/content/dataset/{split_name}/labels", exist_ok=True)

def copy_pair(img_path, split_name):
    label_path = img_path.with_suffix('.txt')
    if not label_path.exists():
        label_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    shutil.copy(img_path, f"/content/dataset/{split_name}/images/{img_path.name}")
    if label_path.exists():
        shutil.copy(label_path, f"/content/dataset/{split_name}/labels/{img_path.stem}.txt")

for img in train_imgs:
    copy_pair(img, 'train')
for img in val_imgs:
    copy_pair(img, 'val')

Upload YOLOv5 dataset


Saving yolo_synthetic_dataset.zip to yolo_synthetic_dataset.zip
200 images found
160 images for training and 40 images for validation


Augmentation (Flip, Rotation, Blur, ...)

In [4]:
def load_labels(label_path):
    if not os.path.exists(label_path):
        return []
    with open(label_path, 'r') as f:
        lines = f.read().strip().splitlines()
    return [list(map(float, l.split())) for l in lines if l]

def save_labels(label_path, labels):
    with open(label_path, 'w') as f:
        for ann in labels:
            cls = int(ann[0])
            f.write(f"{cls} {ann[1]:.6f} {ann[2]:.6f} {ann[3]:.6f} {ann[4]:.6f}\n")

def augment_flip(img, labels):
    img_out = cv2.flip(img, 1)
    new_labels = []
    for ann in labels:
        cls, cx, cy, w, h = ann
        new_labels.append([cls, 1.0 - cx, cy, w, h])
    return img_out, new_labels

def augment_rotate(img, labels, angle):
    H, W = img.shape[:2]
    M = cv2.getRotationMatrix2D((W / 2, H / 2), angle, 1.0)
    img_out = cv2.warpAffine(img, M, (W, H), borderMode=cv2.BORDER_REFLECT)

    new_labels = []
    for ann in labels:
        cls, cx, cy, bw, bh = ann

        cx_px, cy_px = cx * W, cy * H
        bw_px, bh_px = bw * W, bh * H

        corners = np.array([
            [cx_px - bw_px / 2, cy_px - bh_px / 2],
            [cx_px + bw_px / 2, cy_px - bh_px / 2],
            [cx_px + bw_px / 2, cy_px + bh_px / 2],
            [cx_px - bw_px / 2, cy_px + bh_px / 2],
        ])

        ones = np.ones((4, 1))
        corners_h = np.hstack([corners, ones])
        rotated = (M @ corners_h.T).T

        x_min, y_min = rotated[:, 0].min(), rotated[:, 1].min()
        x_max, y_max = rotated[:, 0].max(), rotated[:, 1].max()

        x_min, x_max = np.clip([x_min, x_max], 0, W)
        y_min, y_max = np.clip([y_min, y_max], 0, H)
        new_cx = ((x_min + x_max) / 2) / W
        new_cy = ((y_min + y_max) / 2) / H
        new_bw = (x_max - x_min) / W
        new_bh = (y_max - y_min) / H
        if new_bw > 0 and new_bh > 0:
            new_labels.append([cls, new_cx, new_cy, new_bw, new_bh])
    return img_out, new_labels

def augment_brightness(img, labels, factor):
    img_out = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)
    return img_out, labels

def augment_blur(img, labels, ksize=5):
    img_out = cv2.GaussianBlur(img, (ksize, ksize), 0)
    return img_out, labels

def augment_noise(img, labels, std=15):
    noise = np.random.normal(0, std, img.shape).astype(np.float32)
    img_out = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return img_out, labels

train_img_dir   = "/content/dataset/train/images"
train_label_dir = "/content/dataset/train/labels"
train_image_files = sorted(Path(train_img_dir).glob('*.jpg'))

aug_count = 0

for img_path in train_image_files:
    img = cv2.imread(str(img_path))
    label_path = os.path.join(train_label_dir, img_path.stem + '.txt')
    labels = load_labels(label_path)

    stem = img_path.stem

    aug_img, aug_labels = augment_flip(img, labels)
    cv2.imwrite(f"{train_img_dir}/{stem}_aug_flip.jpg", aug_img)
    save_labels(f"{train_label_dir}/{stem}_aug_flip.txt", aug_labels)

    aug_img, aug_labels = augment_rotate(img, labels, angle=15)
    cv2.imwrite(f"{train_img_dir}/{stem}_aug_rot15.jpg", aug_img)
    save_labels(f"{train_label_dir}/{stem}_aug_rot15.txt", aug_labels)

    aug_img, aug_labels = augment_rotate(img, labels, angle=-15)
    cv2.imwrite(f"{train_img_dir}/{stem}_aug_rot-15.jpg", aug_img)
    save_labels(f"{train_label_dir}/{stem}_aug_rot-15.txt", aug_labels)

    aug_img, aug_labels = augment_brightness(img, labels, factor=0.6)
    cv2.imwrite(f"{train_img_dir}/{stem}_aug_dark.jpg", aug_img)
    save_labels(f"{train_label_dir}/{stem}_aug_dark.txt", aug_labels)

    aug_img, aug_labels = augment_brightness(img, labels, factor=1.4)
    cv2.imwrite(f"{train_img_dir}/{stem}_aug_bright.jpg", aug_img)
    save_labels(f"{train_label_dir}/{stem}_aug_bright.txt", aug_labels)

    aug_img, aug_labels = augment_blur(img, labels, ksize=5)
    cv2.imwrite(f"{train_img_dir}/{stem}_aug_blur.jpg", aug_img)
    save_labels(f"{train_label_dir}/{stem}_aug_blur.txt", aug_labels)

    aug_img, aug_labels = augment_noise(img, labels, std=15)
    cv2.imwrite(f"{train_img_dir}/{stem}_aug_noise.jpg", aug_img)
    save_labels(f"{train_label_dir}/{stem}_aug_noise.txt", aug_labels)

    aug_count += 7

total_train = len(list(Path(train_img_dir).glob('*.jpg')))
print(f"{total_train} images total post-augmentation")

1280 images total post-augmentation


Dataset Configuration File

In [5]:
yaml_content = """path: /content/dataset
train: train/images
val:   val/images

nc: 1
names: ['sprite']
"""

yaml_path = "/content/dataset/dataset.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(yaml_content)

path: /content/dataset
train: train/images
val:   val/images

nc: 1
names: ['sprite']



Training RNA with Medium Weights

In [ ]:
!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 100 \
    --data /content/dataset/dataset.yaml \
    --weights yolov5m.pt \
    --cache

Streaming output truncated to the last 5000 lines.
      69/99      7.33G   0.007552   0.006553          0         66        640:  78% 62/80 [00:28<00:08,  2.18it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      69/99      7.33G   0.007543    0.00654          0         71        640:  79% 63/80 [00:28<00:07,  2.21it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      69/99      7.33G   0.007544   0.006543          0         86        640:  80% 64/80 [00:29<00:07,  2.22it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      69/99      7.33G   0.007542   0

Post-Training Validation

In [ ]:
!python detect.py \
    --weights ./runs/train/exp2/weights/best.pt \
    --img 640 \
    --conf 0.25 \
    --source /content/dataset/val/images

detect: weights=['./runs/train/exp2/weights/best.pt'], source=/content/dataset/val/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-467-g1d62daa3 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 212 layers, 20852934 parameters, 0 gradients, 47.9 GFLOPs
image 1/40 /content/dataset/val/images/image_0001.jpg: 640x640 2 sprites, 26.9ms
image 2/40 /content/dataset/val/images/image_0006.jpg: 640x640 2 sprites, 26.9ms
image 3/40 /content/dataset/val/images/image_0007.jpg: 640x640 2 sprites, 26.9ms
image 4/40 /content/dataset/val/images/image_0008.jpg: 640

Validation Results

In [ ]:
import glob

detected_imgs = glob.glob('/content/yolov5/runs/detect/exp4/*.jpg')
sample = random.sample(detected_imgs, min(6, len(detected_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, img_path in enumerate(sample):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(os.path.basename(img_path), fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

Model Download

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/yolov5_all"
shutil.make_archive(zip_path, 'zip', "/content/yolov5")

size_mb = os.path.getsize(zip_path + ".zip") / (1024 * 1024)
files.download(zip_path + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model Upload

In [6]:
import zipfile
from google.colab import files

print("Upload YOLOv5 model")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, 'r') as z:
    z.extractall("/content/yolov5")

Upload YOLOv5 model


Saving yolov5_all.zip to yolov5_all.zip
